# Burmese narration for Next Editor, made on Colab

The Next Editor studio narrates Burmese lessons with VoxCPM2. This notebook makes that
narration on a Colab GPU instead of the project's hosted one.

1. In the studio: **Workflow → pick the lessons → Narrate on Google Colab → Export requests**.
   It downloads `narration-requests-….json`.
2. Here: **Runtime → Change runtime type → T4 GPU** (or a faster GPU), then **Runtime → Run all**.
   Step 5 asks for the file from step 1.
3. In the studio: **Import audio…** and choose the `audio-….zip` files this notebook wrote
   (or the zip Google Drive makes when you download their folder).

After that, renders of those lessons find every dialog in the browser and ask no speech
service for it.

**A session can end at any moment.** Colab does not promise how long a free session lasts.
With the audio saved to Google Drive (the default), nothing is lost but the last few dialogs:
run the notebook again with the same request file and it goes on from the takes already
written. Keep this tab open while it works, and let step 8 finish before you download the
audio or disconnect.

**A part is not a lesson.** Each `audio-….zip` holds the next few dozen dialogs, so 36 parts
are not 36 lessons. Steps 5, 7 and 8 print which lessons have all their audio, and the same is
kept in `progress.txt` beside the parts. A lesson with some of its audio missing still renders,
but the studio then asks its hosted voice for the missing dialogs.

The notebook works as a batch you run yourself: it starts no server, opens no tunnel and
serves nothing, which is what Colab's rules ask for. Use it only with your own voice, or a
voice you have permission to use.

In [ ]:
#@title 1. Settings { display-mode: "form" }
#@markdown Leave the path empty to upload the request file in step 5. To go on with a pack from an
#@markdown earlier session, paste the path step 5 printed then.
PACK_PATH = ""  #@param {type:"string"}
#@markdown Audio kept only on this runtime is lost when the session ends, finished or not.
SAVE_TO = "Google Drive"  #@param ["Google Drive", "This runtime (lost if the session ends)"]
DRIVE_FOLDER = "next-editor-narration"  #@param {type:"string"}
#@markdown FLAC is lossless and half the size of WAV; the studio reads both.
AUDIO_FORMAT = "flac"  #@param ["flac", "wav"]
#@markdown Takes are written out in parts of at most this many dialogs, so little is lost with a
#@markdown session. A part also ends where a lesson ends, and after ten minutes.
PART_DIALOGS = 25  #@param {type:"integer"}
#@markdown Untested: runs the model in float16, which a T4 computes faster than the model's own
#@markdown bfloat16. Its takes go to a separate `…-float16` folder, so they never mix with the
#@markdown others. Listen to step 6's sample; the studio cannot tell these takes from the hosted
#@markdown voice's once they are imported.
EXPERIMENTAL_FLOAT16 = False  #@param {type:"boolean"}

In [ ]:
#@title 2. Engine settings (generated, nothing to change) { display-mode: "form" }
# What this notebook synthesizes with. A request pack names the same engine, and
# load_pack refuses one that does not: its takes would not be the ones its hashes name.
ENGINE = {
    "model": "openbmb/VoxCPM2",
    "modelRevision": "bffb3df5a29440629464e5e839f4d214c8714c3d",
    "packageVersion": "2.0.3",
    "voiceDesignId": "burmese-educator-v3",
    "cfgValue": 2,
    "inferenceTimesteps": 10,
    "sampleRate": 48000
}
LIMITS = {
    "maxTextChars": 2000,
    "maxSeed": 2147483647,
    "referenceSampleRate": 24000,
    "minReferenceSeconds": 5,
    "maxReferenceSeconds": 20
}
# The delivery every dialog is spoken in, as the hosted synthesizer pins it.
VOICE_DESIGN_PROMPT = "Speak in the reference recording's own delivery — the same speaking rate, the same energy, the same pitch range — as if that speaker had simply kept talking. A Burmese software developer telling a friend how something works, at full conversational speed: alert, direct, and quick to move from one sentence into the next. Never soft, hushed, drowsy, or drawn out; never lecturing or reciting"

In [ ]:
#@title 3. Batch code (generated, nothing to change) { display-mode: "form" }
"""Batch synthesis of a Next Editor narration request pack.

The studio exports the VoxCPM2 speech requests of one or many lessons as a JSON
"request pack" (src/studio/narrationPack/requestPack.ts). This module turns a
pack into audio: one take per dialog, named by the hash the pack gave it, in
zip parts the studio imports (src/studio/narrationPack/audioImport.ts).

It runs as one cell of the Colab notebook beside it (voxcpm2_narration.ipynb,
which is generated from this file), and has no server and no tunnel: a person
runs the notebook and it writes files. Everything here but `make_synthesizer`
is plain Python, so it is tested without a GPU (narration_batch_test.py).

The work can stop at any moment (Colab ends sessions without warning), so it
is written to be run again: finished takes are found in the parts already
written and skipped.
"""

import base64
import binascii
import io
import math
import os
import re
import time
import wave
import zipfile
from dataclasses import dataclass, field, replace

PACK_FORMAT = "next-editor-narration-requests"
PACK_VERSION = 1
# A take's name: the dialog cache's SHA-256 key for it, lowercase hex.
HASH_PATTERN = re.compile(r"[0-9a-f]{64}")
PART_PATTERN = re.compile(r"audio-(\d{4,})\.zip")
AUDIO_FORMATS = {"flac": "FLAC", "wav": "WAV"}
# The studio throws away a take with no sample above this share of full scale.
SILENCE_PEAK = 0.004
# What to do when the output folder stops answering.
STORAGE_HINT = "If the audio goes to Google Drive, run step 5 again: it mounts Drive again"
# Kept beside the parts: which lessons have all their audio, in words.
PROGRESS_FILE = "progress.txt"
PARTS_NOTE = (
    "A part (audio-0001.zip, audio-0002.zip, ...) holds a few dozen dialogs, not one lesson:"
    " the number of parts does not say how many lessons are finished."
)


class PackError(ValueError):
    """The request pack cannot be synthesized as it is; the message says why."""


@dataclass(frozen=True)
class Dialog:
    hash: str
    lesson: str
    text: str
    seed: int
    # Later lessons of the pack that say this same dialog: one take serves them all.
    shared_with: tuple[str, ...] = ()

    @property
    def lessons(self) -> tuple[str, ...]:
        """Every lesson that needs this take, the one the pack files it under first."""
        return (self.lesson, *self.shared_with)


@dataclass(frozen=True)
class Pack:
    created_at: str
    reference_wav: bytes
    dialogs: tuple[Dialog, ...]


def _reference_wav_of(reference: object, limits: dict) -> bytes:
    if not isinstance(reference, dict) or not isinstance(reference.get("wavBase64"), str):
        raise PackError("the pack has no narrator reference recording")
    try:
        reference_wav = base64.b64decode(reference["wavBase64"], validate=True)
    except (binascii.Error, ValueError) as error:
        raise PackError("the narrator reference is not valid base64") from error
    try:
        with wave.open(io.BytesIO(reference_wav), "rb") as reader:
            rate = reader.getframerate()
            if (
                reader.getnchannels() != 1
                or reader.getsampwidth() != 2
                or reader.getcomptype() != "NONE"
                or rate != limits["referenceSampleRate"]
                or rate != reference.get("sampleRate")
            ):
                raise PackError(
                    f"the narrator reference must be a mono {limits['referenceSampleRate']} Hz PCM16 WAV"
                )
            seconds = reader.getnframes() / rate
    except (EOFError, wave.Error) as error:
        raise PackError("the narrator reference is not a valid WAV") from error
    if not limits["minReferenceSeconds"] <= seconds <= limits["maxReferenceSeconds"]:
        raise PackError(
            f"the narrator reference must hold {limits['minReferenceSeconds']}-"
            f"{limits['maxReferenceSeconds']} seconds of speech, not {seconds:.1f}"
        )
    return reference_wav


def _dialog_of(item: object, position: int, limits: dict) -> Dialog:
    where = f"dialog {position}"
    if not isinstance(item, dict):
        raise PackError(f"{where} is not an object")
    digest, text, seed, lesson = item.get("hash"), item.get("text"), item.get("seed"), item.get("lesson")
    if not isinstance(digest, str) or not HASH_PATTERN.fullmatch(digest):
        raise PackError(f"{where} has no valid request hash")
    if not isinstance(text, str) or not text.strip() or len(text.strip()) > limits["maxTextChars"]:
        raise PackError(f"{where} must have 1-{limits['maxTextChars']} characters of text")
    if isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed <= limits["maxSeed"]:
        raise PackError(f"{where} must have a seed between 0 and {limits['maxSeed']}")
    lesson = lesson if isinstance(lesson, str) and lesson else "lesson"
    shared = item.get("sharedWith")
    return Dialog(
        hash=digest,
        lesson=lesson,
        text=text.strip(),
        seed=seed,
        shared_with=_merged((), shared if isinstance(shared, list) else [], lesson),
    )


def _merged(known: tuple[str, ...], more, lesson: str) -> tuple[str, ...]:
    """`known` with the lesson names in `more` it lacks, never `lesson` itself."""
    merged = list(known)
    for name in more:
        if isinstance(name, str) and name and name != lesson and name not in merged:
            merged.append(name)
    return tuple(merged)


def load_pack(data: object, engine: dict, limits: dict) -> Pack:
    """The validated pack. `engine` is what this notebook synthesizes with.

    A take is only the one its hash names when it is made by the engine the
    studio hashed, so a pack that asks for another engine is refused: the
    notebook and the studio are then from different versions of the project.
    """
    if not isinstance(data, dict) or data.get("format") != PACK_FORMAT:
        raise PackError("this file is not a Next Editor narration request pack")
    if data.get("version") != PACK_VERSION:
        raise PackError(
            f"this pack is version {data.get('version')!r} and this notebook reads version {PACK_VERSION}: "
            "open the current notebook from the repository"
        )
    asked = data.get("engine")
    if not isinstance(asked, dict):
        raise PackError("the pack does not say which engine its takes need")
    differences = [
        f"{key}: the pack asks for {asked.get(key)!r}, this notebook has {value!r}"
        for key, value in engine.items()
        if asked.get(key) != value
    ]
    if differences or set(asked) != set(engine):
        raise PackError(
            "this pack and this notebook are from different versions of the project "
            "(open the current notebook from the repository, or export the pack again): "
            + "; ".join(differences or ["the engine settings differ"])
        )
    items = data.get("dialogs")
    if not isinstance(items, list) or not items:
        raise PackError("the pack has no dialogs")
    dialogs: dict[str, Dialog] = {}
    for position, item in enumerate(items, start=1):
        dialog = _dialog_of(item, position, limits)
        first = dialogs.get(dialog.hash)
        if first is None:
            dialogs[dialog.hash] = dialog
        else:
            # One take per hash, and every lesson that asked for it still counts on it.
            dialogs[dialog.hash] = replace(
                first, shared_with=_merged(first.shared_with, dialog.lessons, first.lesson)
            )
    created_at = data.get("createdAt")
    return Pack(
        created_at=created_at if isinstance(created_at, str) else "",
        reference_wav=_reference_wav_of(data.get("reference"), limits),
        dialogs=tuple(dialogs.values()),
    )


def pack_folder_name(pack: Pack) -> str:
    """A folder name for the pack's output that is the same in every session."""
    stamp = re.sub(r"[^0-9A-Za-z]+", "-", pack.created_at).strip("-")
    return f"pack-{stamp}" if stamp else "pack"


def _part_paths(out_dir: str) -> list[tuple[int, str]]:
    parts = []
    for name in os.listdir(out_dir) if os.path.isdir(out_dir) else []:
        match = PART_PATTERN.fullmatch(name)
        if match:
            parts.append((int(match.group(1)), os.path.join(out_dir, name)))
    return sorted(parts)


def finished_hashes(out_dir: str, log=print) -> set[str]:
    """The takes already in the parts written to `out_dir`.

    A part that is not a whole zip is set aside as `.bad`, and its takes are
    synthesized again. A folder that cannot be read at all (a mounted drive that
    dropped) is another matter: that raises, so good parts are not set aside for
    a passing error.
    """
    finished: set[str] = set()
    for _, path in _part_paths(out_dir):
        try:
            # The directory at the end of the file is enough: a part is renamed
            # into place only once it is whole, and reading every take back from
            # a mounted drive would cost minutes at each start.
            with zipfile.ZipFile(path) as part:
                names = part.namelist()
        except zipfile.BadZipFile as error:
            log(f"Setting aside {os.path.basename(path)}: {error}")
            os.replace(path, path + ".bad")
            continue
        except OSError as error:
            raise OSError(f"could not read {path} ({error}). {STORAGE_HINT}") from error
        for name in names:
            stem, _, extension = name.rpartition(".")
            if extension in AUDIO_FORMATS and HASH_PATTERN.fullmatch(stem):
                finished.add(stem)
    return finished


class LessonProgress:
    """How many of each lesson's dialogs have a take written, in the pack's order.

    The parts in a folder do not show this: a part holds the next few dozen
    dialogs, whichever lessons they belong to. A lesson can be rendered without
    a speech service only once every one of its dialogs has a take.
    """

    def __init__(self, dialogs, finished=()):
        dialogs = list(dialogs)
        self._lessons_of: dict[str, tuple[str, ...]] = {}
        self._total: dict[str, int] = {}
        self._written: dict[str, int] = {}
        self._hashes: set[str] = set()
        # The pack's order is the order of the lessons its dialogs are filed under; a
        # lesson that only shares dialogs with earlier ones comes after them.
        for dialog in dialogs:
            self._total.setdefault(dialog.lesson, 0)
        for dialog in dialogs:
            # A dialog two lessons say is one take, and each of them needs it.
            self._lessons_of[dialog.hash] = dialog.lessons
            for lesson in dialog.lessons:
                self._total[lesson] = self._total.get(lesson, 0) + 1
        self._written = dict.fromkeys(self._total, 0)
        self.add(finished)

    def add(self, hashes) -> list[str]:
        """Count these takes as written; returns the lessons they completed."""
        completed = []
        for digest in hashes:
            if digest in self._hashes:
                continue
            self._hashes.add(digest)
            for lesson in self._lessons_of.get(digest, ()):
                self._written[lesson] += 1
                if self._written[lesson] == self._total[lesson]:
                    completed.append(lesson)
        return completed

    def completes(self, hashes) -> bool:
        """Whether writing these takes would complete a lesson."""
        staged: dict[str, int] = {}
        for digest in set(hashes) - self._hashes:
            for lesson in self._lessons_of.get(digest, ()):
                staged[lesson] = staged.get(lesson, 0) + 1
        return any(
            count == self._total[lesson] - self._written[lesson] for lesson, count in staged.items()
        )

    @property
    def lessons(self) -> list[tuple[str, int, int]]:
        """(lesson, dialogs with a take, dialogs) for every lesson, in the pack's order."""
        return [(lesson, self._written[lesson], total) for lesson, total in self._total.items()]

    @property
    def complete(self) -> int:
        """The number of lessons with a take for every dialog."""
        return sum(1 for _, written, total in self.lessons if written == total)


def counted(number: int, one: str) -> str:
    """`number` with its noun, singular for one: "1 dialog", "2 dialogs"."""
    return f"{number} {one}" if number == 1 else f"{number} {one}s"


def describe_lessons(progress: LessonProgress) -> str:
    """Which lessons have all their audio, for the notebook to print.

    A dialog two lessons share counts for each of them, so the dialogs counted
    here can be more than the takes in the parts.
    """
    lessons = progress.lessons
    if len(lessons) == 1:
        lesson, written, total = lessons[0]
        if written == total:
            return f"The one lesson, {lesson}, has all its audio ({counted(total, 'dialog')})."
        return f"The one lesson, {lesson}, has {written} of its {counted(total, 'dialog')}."
    written = sum(count for _, count, _ in lessons)
    total = sum(count for _, _, count in lessons)
    if progress.complete == len(lessons):
        return f"All {len(lessons)} lessons have all their audio ({counted(total, 'dialog')})."
    lines = [
        f"{progress.complete} of {len(lessons)} lessons have all their audio"
        f" ({written} of {counted(total, 'dialog')})."
    ]
    # Lessons in a row that are complete, or that have nothing, share a line.
    runs: list[tuple[str, list[str]]] = []
    for lesson, count, of in lessons:
        if 0 < count < of:
            runs.append((f"{count} of {counted(of, 'dialog')}", [lesson]))
            continue
        state = "complete" if count == of else "no audio yet"
        if runs and runs[-1][0] == state:
            runs[-1][1].append(lesson)
        else:
            runs.append((state, [lesson]))
    for state, names in runs:
        span = names[0] if len(names) == 1 else f"{names[0]} ... {names[-1]} ({len(names)} lessons)"
        lines.append(f"  {state}: {span}")
    return "\n".join(lines)


def write_progress(out_dir: str, progress: LessonProgress) -> None:
    """Leave the lessons' progress beside the parts, for whoever opens the folder.

    Never raises: the parts are what matters, and a folder that is gone is
    reported by the next part that cannot be written.
    """
    text = "\n".join(
        [
            describe_lessons(progress),
            "",
            PARTS_NOTE,
            "Import every part into the studio. Its Narrate on Google Colab section then shows"
            " which of the lessons you pick still lack audio.",
            "",
        ]
    )
    path = os.path.join(out_dir, PROGRESS_FILE)
    try:
        os.makedirs(out_dir, exist_ok=True)
        with open(path + ".partial", "w", encoding="utf-8") as target:
            target.write(text)
        os.replace(path + ".partial", path)
    except OSError:
        pass


def pack_state(pack: Pack, out_dir: str, log=print) -> tuple[list[Dialog], LessonProgress]:
    """The dialogs with no take in `out_dir` yet, and the lessons' progress.

    Both come from one reading of the parts: on a mounted drive each reading
    opens every part.
    """
    finished = finished_hashes(out_dir, log)
    pending = [dialog for dialog in pack.dialogs if dialog.hash not in finished]
    return pending, LessonProgress(pack.dialogs, finished)


def pending_dialogs(pack: Pack, out_dir: str, log=print) -> list[Dialog]:
    """The pack's dialogs that have no take in `out_dir` yet, in the pack's order."""
    return pack_state(pack, out_dir, log)[0]


def write_part(out_dir: str, takes: list[tuple[str, bytes]], extension: str) -> str:
    """Write these takes as the next `audio-NNNN.zip` part; returns its path.

    The part appears under its name only once it is whole, so a session that
    ends mid-write leaves no half part behind for the studio to trip over.
    """
    os.makedirs(out_dir, exist_ok=True)
    parts = _part_paths(out_dir)
    number = parts[-1][0] + 1 if parts else 1
    path = os.path.join(out_dir, f"audio-{number:04d}.zip")
    partial = path + ".partial"
    # The audio is already compressed (FLAC) or does not compress (PCM): store it.
    with zipfile.ZipFile(partial, "w", zipfile.ZIP_STORED) as part:
        for digest, audio in takes:
            part.writestr(f"{digest}.{extension}", audio)
    os.replace(partial, path)
    return path


def duration(seconds: float) -> str:
    minutes = int(round(seconds / 60))
    hours, minutes = divmod(minutes, 60)
    return f"{hours} h {minutes:02d} min" if hours else f"{minutes} min"


@dataclass
class RunSummary:
    synthesized: int = 0
    # Characters of text in the synthesized dialogs: what the time left is judged by.
    characters: int = 0
    audio_seconds: float = 0.0
    work_seconds: float = 0.0
    failed: list[tuple[Dialog, str]] = field(default_factory=list)
    parts: list[str] = field(default_factory=list)
    # Why the run ended before the last dialog, or None.
    stopped: str | None = None

    @property
    def speed(self) -> float | None:
        """Seconds of work per second of audio, or None before the first take."""
        return self.work_seconds / self.audio_seconds if self.audio_seconds else None


def seconds_for(characters: int, summary: RunSummary) -> float:
    """The time `characters` more of text take at the pace of the run so far."""
    return characters * summary.work_seconds / summary.characters if summary.characters else 0.0


def run_batch(
    dialogs: list[Dialog],
    out_dir: str,
    synthesize,
    *,
    extension: str = "flac",
    part_dialogs: int = 25,
    part_seconds: float = 600.0,
    max_consecutive_failures: int = 5,
    lessons: LessonProgress | None = None,
    clock=time.monotonic,
    log=print,
) -> RunSummary:
    """Synthesize `dialogs` into zip parts in `out_dir`.

    `synthesize(dialog)` returns (encoded audio, seconds of audio). A dialog it
    raises on is reported and skipped; several in a row end the run, since the
    cause is then the runtime (out of memory, a lost GPU) and not the dialog.
    Takes are written out every `part_dialogs` dialogs or `part_seconds`
    seconds, and when the run is interrupted, so little is lost with a session.

    With `lessons` (the pack's progress so far), a part is also written when a
    lesson's last take is made, each lesson is named in the log once all its
    audio is written, and the progress is kept in a file beside the parts.
    """
    if extension not in AUDIO_FORMATS:
        raise ValueError(f"unknown audio format {extension!r}")
    summary = RunSummary()
    staged: list[tuple[str, bytes]] = []
    last_write = clock()
    total_characters = sum(len(dialog.text) for dialog in dialogs)
    consecutive_failures = 0

    def write_staged() -> bool:
        """Write the staged takes out. False when the folder cannot be written."""
        nonlocal last_write
        if staged:
            try:
                summary.parts.append(write_part(out_dir, staged, extension))
            except OSError as error:
                # The takes are gone with this run: say so, and do not count them.
                summary.synthesized -= len(staged)
                summary.stopped = (
                    f"could not write to {out_dir} ({error}); the last {len(staged)} take(s)"
                    f" were not saved. {STORAGE_HINT}"
                )
                staged.clear()
                return False
            # The takes are on disk: nothing below may leave them staged, or an
            # interrupt there would write them out a second time on the way out.
            written = [digest for digest, _ in staged]
            staged.clear()
            if lessons is not None:
                for lesson in lessons.add(written):
                    log(
                        f"{lesson}: all its audio is written"
                        f" ({lessons.complete} of {counted(len(lessons.lessons), 'lesson')})."
                    )
                write_progress(out_dir, lessons)
        last_write = clock()
        return True

    try:
        for position, dialog in enumerate(dialogs, start=1):
            started = clock()
            try:
                audio, seconds = synthesize(dialog)
            except Exception as error:  # noqa: BLE001 - one bad dialog must not end the batch
                summary.failed.append((dialog, f"{type(error).__name__}: {error}"))
                log(f"[{position}/{len(dialogs)}] {dialog.lesson}: failed, {type(error).__name__}: {error}")
                consecutive_failures += 1
                if consecutive_failures >= max_consecutive_failures:
                    summary.stopped = f"{consecutive_failures} dialogs failed in a row"
                    break
                continue
            consecutive_failures = 0
            took = clock() - started
            staged.append((dialog.hash, audio))
            summary.synthesized += 1
            summary.audio_seconds += seconds
            summary.work_seconds += took
            summary.characters += len(dialog.text)
            remaining = seconds_for(total_characters - summary.characters, summary)
            log(
                f"[{position}/{len(dialogs)}] {dialog.lesson}: {seconds:.1f} s of audio in {took:.1f} s"
                f" ({summary.speed:.1f}x) · about {duration(remaining)} left"
            )
            if (
                len(staged) >= part_dialogs
                or clock() - last_write >= part_seconds
                or (lessons is not None and lessons.completes(digest for digest, _ in staged))
            ):
                if not write_staged():
                    break
    except KeyboardInterrupt:
        summary.stopped = "interrupted"
    finally:
        write_staged()
    return summary


def describe(summary: RunSummary, out_dir: str) -> str:
    """What a run did, for the notebook to print."""
    lines = [
        f"Synthesized {summary.synthesized} dialogs: {duration(summary.audio_seconds)} of audio"
        f" in {duration(summary.work_seconds)}"
        + (f" ({summary.speed:.1f} s of work per second of audio)." if summary.speed else "."),
        f"Wrote {len(summary.parts)} part(s) to {out_dir}",
    ]
    if summary.failed:
        lines.append(f"{len(summary.failed)} dialog(s) failed and have no take:")
        lines += [f"  {dialog.lesson} {dialog.hash[:12]}: {reason}" for dialog, reason in summary.failed[:10]]
    if summary.stopped:
        lines.append(f"Stopped early: {summary.stopped}. Then run this cell again to go on from here.")
    return "\n".join(lines)


def make_synthesizer(
    model,
    *,
    torch,
    soundfile,
    reference_path: str,
    prompt: str,
    engine: dict,
    extension: str = "flac",
):
    """`synthesize(dialog)` for `run_batch`, as the project's hosted synthesizer does it.

    The same call as integrations/modal/voxcpm2_tts.py: the delivery prompt in
    parentheses before the text, the narrator reference, the seed set before
    each dialog. A GPU of another kind does not reproduce the hosted take
    sample for sample; it is a take of the same request all the same.
    """
    sample_rate = int(engine["sampleRate"])

    def synthesize(dialog: Dialog) -> tuple[bytes, float]:
        torch.manual_seed(dialog.seed)
        wav = model.generate(
            text=f"({prompt}) {dialog.text}",
            reference_wav_path=reference_path,
            cfg_value=float(engine["cfgValue"]),
            inference_timesteps=int(engine["inferenceTimesteps"]),
        )
        if len(wav) == 0:
            raise ValueError("the model returned no audio")
        peak = float(abs(wav).max())
        # NaN compares false with everything, so it would pass the silence test.
        if not math.isfinite(peak):
            raise ValueError("the model returned samples that are not numbers")
        if peak <= SILENCE_PEAK:
            raise ValueError("the model returned silence")
        encoded = io.BytesIO()
        soundfile.write(encoded, wav, sample_rate, format=AUDIO_FORMATS[extension], subtype="PCM_16")
        return encoded.getvalue(), len(wav) / sample_rate

    return synthesize

In [ ]:
#@title 4. Install VoxCPM
import importlib.metadata
import subprocess
import sys


def installed(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


if installed("voxcpm") != ENGINE["packageVersion"]:
    # Colab's own PyTorch stays: VoxCPM asks only for 2.5 or newer, and replacing a
    # GPU build of PyTorch costs minutes and can leave one that does not see the GPU.
    pins = [f"{name}=={installed(name)}" for name in ("torch", "torchaudio") if installed(name)]
    with open("voxcpm-constraints.txt", "w", encoding="utf-8") as constraints:
        constraints.write("\n".join(pins))
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            f"voxcpm=={ENGINE['packageVersion']}",
            "--constraint",
            "voxcpm-constraints.txt",
        ],
        check=True,
    )
print("voxcpm", installed("voxcpm"), "· torch", installed("torch"))

In [ ]:
#@title 5. Open the request pack
import json
import os
import shutil
import tempfile

IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = IN_COLAB and SAVE_TO == "Google Drive"

# Before the pack is opened: a pack from an earlier session is on Drive.
if IN_COLAB and (USE_DRIVE or PACK_PATH.strip().startswith("/content/drive")):
    from google.colab import drive

    drive.mount("/content/drive")



def require_output_folder():
    """Stop before work is written to a folder that only looks like Google Drive."""
    if USE_DRIVE and not os.path.ismount("/content/drive"):
        raise SystemExit("Google Drive is not mounted. Run step 5 again, then this cell.")


if PACK_PATH.strip():
    pack_path = PACK_PATH.strip()
    if not os.path.isfile(pack_path):
        raise SystemExit(
            f"{pack_path} was not found. Check the path, or empty PACK_PATH and upload the request file."
        )
elif IN_COLAB:
    from google.colab import files

    print("Choose the narration-requests-….json file the studio exported.")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file was chosen. Run this cell again.")
    pack_path = next(iter(uploaded))
else:
    raise SystemExit("Set PACK_PATH in step 1 to the request pack's path.")

with open(pack_path, encoding="utf-8") as source:
    pack = load_pack(json.load(source), ENGINE, LIMITS)

if USE_DRIVE:
    output_root = os.path.join("/content/drive/MyDrive", DRIVE_FOLDER.strip() or "next-editor-narration")
else:
    output_root = os.path.abspath("narration")
# The float16 experiment gets a folder of its own: its takes must never count as
# finished for a run with the model's own number format, or the other way round.
OUT_DIR = os.path.join(
    output_root, pack_folder_name(pack) + ("-float16" if EXPERIMENTAL_FLOAT16 else "")
)
os.makedirs(OUT_DIR, exist_ok=True)

# The pack is kept beside its audio, so a later session can open it from there.
saved_pack = os.path.join(OUT_DIR, "requests.json")
if os.path.abspath(pack_path) != os.path.abspath(saved_pack):
    shutil.copyfile(pack_path, saved_pack)

REFERENCE_PATH = os.path.join(tempfile.gettempdir(), "narrator-reference.wav")
with open(REFERENCE_PATH, "wb") as reference_file:
    reference_file.write(pack.reference_wav)

todo, progress = pack_state(pack, OUT_DIR)
write_progress(OUT_DIR, progress)
print(
    f"{counted(len(pack.dialogs), 'dialog')} of {counted(len(progress.lessons), 'lesson')};"
    f" {len(todo)} still to synthesize."
)
print(describe_lessons(progress))
print(f"The audio goes to: {OUT_DIR}")
if USE_DRIVE:
    print(f"To go on in a later session, set PACK_PATH in step 1 to: {saved_pack}")
else:
    print(
        "This runtime's disk goes when the session ends: every take not downloaded in step 8 by"
        " then is lost, and a later session starts from the first dialog. Choose Google Drive in"
        " step 1 to keep them."
    )

In [ ]:
#@title 6. Load the model and try two dialogs
import io
import zipfile

import soundfile
import torch
from huggingface_hub import snapshot_download
from voxcpm import VoxCPM

require_output_folder()
if not torch.cuda.is_available():
    raise SystemExit("This runtime has no GPU. Runtime > Change runtime type > T4 GPU, then run all again.")
print("GPU:", torch.cuda.get_device_name(0))

MODEL_DIR = snapshot_download(
    repo_id=ENGINE["model"], revision=ENGINE["modelRevision"], local_dir="VoxCPM2"
)
config_path = os.path.join(MODEL_DIR, "config.json")
original_config_path = config_path + ".original"
if EXPERIMENTAL_FLOAT16:
    if not os.path.exists(original_config_path):
        shutil.copyfile(config_path, original_config_path)
    with open(original_config_path, encoding="utf-8") as source:
        model_config = json.load(source)
    model_config["dtype"] = "float16"
    with open(config_path, "w", encoding="utf-8") as target:
        json.dump(model_config, target)
elif os.path.exists(original_config_path):
    shutil.copyfile(original_config_path, config_path)

# Eager inference without the denoiser, as the hosted synthesizer runs it: the compiled
# path spends minutes warming up at every start, and the narrator reference is clean.
model = VoxCPM.from_pretrained(MODEL_DIR, load_denoiser=False, optimize=False)
if int(model.tts_model.sample_rate) != ENGINE["sampleRate"]:
    raise SystemExit(f"Expected {ENGINE['sampleRate']} Hz audio from VoxCPM2, got {model.tts_model.sample_rate} Hz.")

synthesize = make_synthesizer(
    model,
    torch=torch,
    soundfile=soundfile,
    reference_path=REFERENCE_PATH,
    prompt=VOICE_DESIGN_PROMPT,
    engine=ENGINE,
    extension=AUDIO_FORMAT,
)

trial = run_batch(todo[:2], OUT_DIR, synthesize, extension=AUDIO_FORMAT, part_dialogs=1)
if todo:
    print(describe(trial, OUT_DIR))
else:
    print("Every dialog already has a take: nothing to try.")
todo, progress = pack_state(pack, OUT_DIR)
# The trial wrote parts too: the progress kept beside them must count those.
write_progress(OUT_DIR, progress)
if trial.synthesized:
    rest = seconds_for(sum(len(dialog.text) for dialog in todo), trial)
    print(f"At this pace the other {len(todo)} dialogs take about {duration(rest)}.")
    # Listen before the long run: this is the voice every dialog gets.
    from IPython.display import Audio, display

    with zipfile.ZipFile(trial.parts[0]) as part:
        samples, rate = soundfile.read(io.BytesIO(part.read(part.namelist()[0])))
    display(Audio(samples, rate=rate))

In [ ]:
#@title 7. Synthesize everything
# Stop it whenever you like (Runtime > Interrupt execution): the takes made so far are
# written out, and running this cell again goes on from there.
require_output_folder()
todo, progress = pack_state(pack, OUT_DIR)
if todo:
    summary = run_batch(
        todo,
        OUT_DIR,
        synthesize,
        extension=AUDIO_FORMAT,
        part_dialogs=PART_DIALOGS,
        lessons=progress,
    )
    print(describe(summary, OUT_DIR))
else:
    print("Every dialog already has a take.")
print(describe_lessons(progress))

In [ ]:
#@title 8. Get the audio
require_output_folder()
part_names = sorted(name for name in os.listdir(OUT_DIR) if PART_PATTERN.fullmatch(name))
megabytes = sum(os.path.getsize(os.path.join(OUT_DIR, name)) for name in part_names) / 1e6
left, progress = pack_state(pack, OUT_DIR)
write_progress(OUT_DIR, progress)
print(f"{len(part_names)} part(s), {megabytes:.0f} MB, in {OUT_DIR}")
print(describe_lessons(progress))
if left:
    print(f"{len(left)} dialog(s) still have no take: run step 7 again.")
    print(PARTS_NOTE)
if USE_DRIVE:
    # Drive takes files in the background; this waits until it has them all.
    print("Sending the last parts to Google Drive. Wait for this cell before you download or disconnect.")
    drive.flush_and_unmount()
    print("Google Drive has every part. To synthesize more, run step 5 again first: it mounts Drive again.")
elif IN_COLAB:
    from google.colab import files

    for name in part_names:
        files.download(os.path.join(OUT_DIR, name))
print(
    "In the studio: Workflow > Narrate on Google Colab > Import audio…, then choose the"
    " audio-….zip files (or the zip Google Drive makes when you download this folder)."
    " Pick the lessons there to see which of them still lack audio before you render."
)